Reusing Prompt Template using LangChain

LLM --> Produces default String type data

In [2]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.environ.get("GROQ_API"),
)

In [ ]:
def ask_groq(prompt):
   
    response = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="llama-3.3-70b-versatile",
        stream=False,
    )
    
    # Adjust this based on the actual structure of the response
    return response.choices[0].message.content

In [6]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [7]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [9]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [10]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [11]:
customer_msg = prompt_template.format_prompt(
    style=customer_style,
    text=customer_email,
)

In [16]:
customer_msg.messages[0].content

"Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n"

In [17]:
customer_response = ask_groq(customer_msg.messages[0].content)
customer_response

ChatCompletion(id='chatcmpl-c3766a11-e98e-45a4-a61e-e403a5493a57', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I'm having a bit of a frustrating issue with my blender. The lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning up the mess. I would greatly appreciate your assistance with this problem as soon as possible.", role='assistant', executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1745264064, model='llama-3.3-70b-versatile', object='chat.completion', system_fingerprint='fp_9a8b91ba77', usage=CompletionUsage(completion_tokens=61, prompt_tokens=117, total_tokens=178, completion_time=0.221818182, prompt_time=0.007340443, queue_time=0.219700431, total_time=0.229158625), usage_breakdown=UsageBreakdown(models=None), x_groq={'id': 'req_01jscxgdxtfs19fnjrgx4bn4kz'})


"I'm having a bit of a frustrating issue with my blender. The lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning up the mess. I would greatly appreciate your assistance with this problem as soon as possible."

In [19]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

service_style = """American English \
in a calm and respectful tone
"""

In [20]:
service_msg = prompt_template.format_prompt(
    style=service_style,
    text=service_reply,
)

In [21]:
service_msg.messages[0].content

"Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!\n```\n"

In [23]:
service_response = ask_groq(service_msg.messages[0].content)
service_response

ChatCompletion(id='chatcmpl-d4fc7b59-6a82-4aee-b4b0-fa7064c9d1cf', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="```We appreciate you reaching out to us, valued customer. Regarding the warranty on your kitchen blender, we regret to inform you that it does not cover cleaning expenses due to misuse. Unfortunately, operating the blender without the lid can cause damage and create a mess. We understand this may be inconvenient, and we're here to help. For future reference, please ensure the lid is securely in place before use to prevent such incidents. If you have any further questions or concerns, please don't hesitate to contact us. We're committed to providing you with excellent service and support.```", role='assistant', executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1745264262, model='llama-3.3-70b-versatile', object='chat.completion', system_fingerprint='fp_3f3b593e33', usage=CompletionUsage(

"```We appreciate you reaching out to us, valued customer. Regarding the warranty on your kitchen blender, we regret to inform you that it does not cover cleaning expenses due to misuse. Unfortunately, operating the blender without the lid can cause damage and create a mess. We understand this may be inconvenient, and we're here to help. For future reference, please ensure the lid is securely in place before use to prevent such incidents. If you have any further questions or concerns, please don't hesitate to contact us. We're committed to providing you with excellent service and support.```"

Structuring Output into desired Format - JSON

In [24]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [25]:
prompt_template2 = ChatPromptTemplate.from_template(review_template)

In [26]:
customer_msg = prompt_template2.format_prompt(
    text=customer_review,
)

In [27]:
answer = ask_groq(customer_msg.messages[0].content)
answer

ChatCompletion(id='chatcmpl-c0d93ae1-f073-4b3b-86be-2193dade7c2c', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```\n{\n  "gift": True,\n  "delivery_days": 2,\n  "price_value": ["It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features."]\n}\n```', role='assistant', executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1745265516, model='llama-3.3-70b-versatile', object='chat.completion', system_fingerprint='fp_6507bcfb6f', usage=CompletionUsage(completion_tokens=52, prompt_tokens=246, total_tokens=298, completion_time=0.189090909, prompt_time=0.015860481, queue_time=0.221132513, total_time=0.20495139), usage_breakdown=UsageBreakdown(models=None), x_groq={'id': 'req_01jscywqj8fdzv5h8j5jwq9gpz'})


'```\n{\n  "gift": True,\n  "delivery_days": 2,\n  "price_value": ["It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features."]\n}\n```'

In [28]:
type(answer)

str

Creating JSON (python dictionary) type data using LangChain

In [31]:
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

In [32]:
gift_schema = ResponseSchema(
    name="gift",
    description="Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.",
)
delivery_days_schema = ResponseSchema(
    name="delivery_days",
    description="How many days did it take for the product to arrive? If this information is not found, output -1.",
)
price_value_schema = ResponseSchema(
    name="price_value",
    description="Extract any sentences about the value or price, and output them as a comma separated Python list.",
)

response_schemas = [
    gift_schema,
    delivery_days_schema,
    price_value_schema,
]

In [33]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)


In [34]:
format_instructions = output_parser.get_format_instructions()
format_instructions

'The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"gift": string  // Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\t"delivery_days": string  // How many days did it take for the product to arrive? If this information is not found, output -1.\n\t"price_value": string  // Extract any sentences about the value or price, and output them as a comma separated Python list.\n}\n```'

In [ ]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)

In [36]:
messages[0].content

'For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\ntext: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife\'s anniversary present. I think my wife liked it so much she was speechless. So far I\'ve been the only one using it, and I\'ve been using it every other morning to clear the leaves on our lawn. It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features.\n\n\nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "

In [40]:
response = ask_groq(messages[0].content)
response

ChatCompletion(id='chatcmpl-a01415ff-d8d5-4814-a913-d8cb64339daa', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```json\n{\n\t"gift": "True",\n\t"delivery_days": "2",\n\t"price_value": "It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features."\n}\n```', role='assistant', executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1745267927, model='llama-3.3-70b-versatile', object='chat.completion', system_fingerprint='fp_3f3b593e33', usage=CompletionUsage(completion_tokens=54, prompt_tokens=349, total_tokens=403, completion_time=0.196363636, prompt_time=0.029432616, queue_time=0.220535482, total_time=0.225796252), usage_breakdown=UsageBreakdown(models=None), x_groq={'id': 'req_01jsd16a60f6vvprbdmgjrhra9'})


'```json\n{\n\t"gift": "True",\n\t"delivery_days": "2",\n\t"price_value": "It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features."\n}\n```'

In [42]:
output_dictionary = output_parser.parse(response)

In [43]:
output_dictionary

{'gift': 'True',
 'delivery_days': '2',
 'price_value': "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."}

In [45]:
output_dictionary['price_value']

"It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."

In [46]:
type(output_dictionary)

dict